In [ ]:
# Importer les données de data.gouv et les afficher
import pandas as pd

df = pd.read_csv(
    "data/tonnes-kilometres-de-marchandises-transportees-par-region-darrivee-transport-marchandise-destination-region.csv"
)

columns_names_mapping = {
    "tonnes_kilometres_de_marchandises_transportees_par_region_d_arrivee_transport_marchandise_destination_region": "tonnes_km",
    "date_mesure.year": "annee",
    "date_mesure": "date",
}

df = df.rename(columns=columns_names_mapping)

In [ ]:
list_regions_francaises = [
    "Auvergne-Rhône-Alpes",
    "Bourgogne-Franche-Comté",
    "Bretagne",
    "Centre-Val de Loire",
    "Corse",
    "Grand Est",
    "Hauts-de-France",
    "Île-de-France",
    "Normandie",
    "Nouvelle-Aquitaine",
    "Occitanie",
    "Pays de la Loire",
    "Provence-Alpes-Côte d'Azur",
]

In [ ]:
df["pays"] = df["libelle_region"].apply(
    lambda x: "France" if x in list_regions_francaises else x
)
df["annee"] = pd.to_datetime(df["annee"]).dt.year

In [ ]:
df

In [ ]:
df_route = (
    df[(df["mode_transport"] == "routier") & (df["annee"] >= 2017)]
    .groupby(["pays", "date"])
    .agg({"tonnes_km": "sum"})
    .reset_index()
)

In [ ]:
# Pivot: pour avoir la tonnes_km par pays en colonne et drop les colonnes avec des valeurs manquantes
df_route_pivot = df_route.pivot(index="date", columns="pays", values="tonnes_km")
df_route_pivot = df_route_pivot.dropna(axis=1)

In [ ]:
from sklearn.linear_model import LinearRegression

df_routier_fr = df_route_pivot[["France"]].reset_index()
df_routier_fr["date"] = pd.to_datetime(df_routier_fr["date"])
# Nous allons utiliser la régression linéaire pour calculer la tendance du transport routier en France

In [ ]:
df_routier_fr["annee"] = df_routier_fr["date"].dt.year

In [ ]:
X = df_routier_fr["annee"].values.reshape(-1, 1)
y = df_routier_fr["France"].values

model = LinearRegression()

model.fit(X, y)

In [ ]:
a = model.coef_[0]
b = model.intercept_

In [ ]:
import numpy as np


y = model.predict(X)

In [ ]:
# Plot the data using plotly
import plotly.express as px

fig = px.bar(
    df_routier_fr,
    x="annee",
    y="France",
    title="Tonnes-kilomètres de marchandises transportées par route en France",
    labels={"value": "Tonnes-km", "date": "Année"},
)

fig.add_scatter(
    x=df_routier_fr["annee"],
    y=model.predict(df_routier_fr["annee"].values.reshape(-1, 1)),
    mode="lines",
    line=dict(color="red", width=3),
    name="Tendance",
)

fig.show()